In [12]:
import pandas as pd
import numpy as np
import sqlite3
import os
import glob

#设置pandas显示选项
pd.set_option('display.max_columns', None)
pd.set_option("display.width",1000)
#设置文件路径

data_path = './data/'
#读取所有数据
csv_files = glob.glob(data_path + '*.csv')
#建立字典
dataframes = {}
#遍历数据
for file in csv_files:
    name = os.path.basename(file).replace('.csv', '')  #替换将后缀替换为空
    dataframes[name] = pd.read_csv(file)

In [13]:
#提取用户表和订单表
customers = dataframes["olist_customers_dataset"]
orders = dataframes["olist_orders_dataset"]

#查看用户表
print("用户表")
customers.head()
#查看订单数
print("订单表")
orders.head()

用户表
订单表


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [14]:
#查看订单数
print("订单表")
orders.head()

订单表


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [15]:
print("customer_id 数量：",
      customers["customer_id"].nunique())

print("customer_unique_id 数量：",
      customers["customer_unique_id"].nunique())

customer_id 数量： 99441
customer_unique_id 数量： 96096


In [16]:
#统计真实用户的订单次数
user_orders = orders.groupby(
    "customer_id"
)["order_id"].count()#以c_id分组，统计o_id这列不为空的数
user_orders.head()

customer_id
00012a2ce6f8dcda20d059ce98491703    1
000161a058600d5901f007fab4c27140    1
0001fd6190edaaf884bcaf3d49edf079    1
0002414f95344307404f0ace7a26f1d5    1
000379cdec625522490c315e70c7a9fb    1
Name: order_id, dtype: int64

In [17]:
#关联真实用户编号
user_data= pd.merge(
    orders,
    customers[["customer_id", "customer_unique_id"]],
    on = "customer_id",
    how = "left"
)#两表关于C-id左连接，生成新表user_data
print(user_data[[
    "customer_id",
    "customer_unique_id",
    "order_id",
]].head())

                        customer_id                customer_unique_id                          order_id
0  9ef432eb6251297304e76186b10a928d  7c396fd4830fd04220f754e42b4e5bff  e481f51cbdc54678b7cc49136f2d6af7
1  b0830fb4747a6c6d20dea0b8c802d7ef  af07308b275d755c9edb36a90c618231  53cdb2fc8bc7dce0b6741e2150273451
2  41ce2a54c0b03bf3443c3d931a367089  3a653a41f6f9fc3d2a113cf8398680e8  47770eb9100c2d0c44946d9cf07ec65d
3  f88197465ea7920adcdbec7375364d82  7c142cf63193a1473d2e66489a9ae977  949d5b44dbf5de918fe9c16f97b45f8a
4  8ab97904e6daea8866dbdbc4fb7aad2c  72632f0f9dd73dfee390c9b22eb56dd6  ad21c59c0840e6cb83a9ceb5573f8159


In [18]:
#统计每位真实用户的订单
user_purchase_cont = user_data.groupby(
    "customer_unique_id"
)["order_id"].count()
user_purchase_cont.head()
#左侧真实用户编号，右侧订单数

customer_unique_id
0000366f3b9a7992bf8c76cfdf3221e2    1
0000b849f77a49e4a4ce2b2a4ca5be3f    1
0000f46a3911fa3c0805444483337064    1
0000f6ccb0745a6a4b88665a16c9f078    1
0004aac84e0df4da2b147fca70cf8255    1
Name: order_id, dtype: int64

In [19]:
#统计复购用户数
repeat_user = user_purchase_cont[
    user_purchase_cont > 1#筛选用户
]
print("复购用户数量：",len(repeat_user))
print("复购用户订单次数分布")
print(repeat_user.value_counts().sort_index())#前者统计不同订单分别出现多少次，后者按索引从小到大排序

复购用户数量： 2997
复购用户订单次数分布
order_id
2     2745
3      203
4       30
5        8
6        6
7        3
9        1
17       1
Name: count, dtype: int64


In [20]:
#计算用户复购率：复购率=（复购用户/真实用户）*100%
repeat_rate = len(repeat_user)/customers["customer_unique_id"].nunique()#统计不重复值
print(repeat_rate * 100 ,"%")

3.1187562437562435 %


In [21]:
#统计用户购买次数分布
print("mean",user_purchase_cont.mean())
print("max",user_purchase_cont.max())
print("min",user_purchase_cont.min())

mean 1.0348089410589412
max 17
min 1
